# EMPIEZA NICOLE

# Learning Urban Crimes Representation

## Visualización e interpretación del espacio latente

Input:  embeddings_h3_optimal.csv, clusters_h3_optimal.csv, h3_metadata.csv,
        firmas_h3.csv

Output:
  A) Visualización abstracta del espacio latente (UMAP + t-SNE)
     - vis_umap_clusters.html (interactivo)
     - vis_tsne_clusters.html (interactivo)
  B) Visualización geográfica sobre mapa de CDMX
     - mapa_clusters.html (hexágonos coloreados por cluster)
     - mapa_violencia.html (hexágonos coloreados por ratio de violencia)
     - mapa_intensidad.html (hexágonos coloreados por intensidad)

### Paquetes

In [15]:
import pandas as pd
import numpy as np
import h3
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

import umap

# Gráficos
import folium
from folium.plugins import FloatImage
import branca.colormap as cm

import json

### Ejecución

#### Cargar datos

In [7]:
# Cargar datos
embeddings = pd.read_csv("../data/results/embeddings_h3_optimal.csv", index_col='h3_id')
clusters = pd.read_csv("../data/results/clusters_h3_optimal.csv", index_col='h3_id')
metadata = pd.read_csv("../data/auxiliar/h3_metadata.csv", index_col='h3_id')
firmas = pd.read_csv("../data/auxiliar/firmas_h3.csv", index_col='h3_id')

# Unir todo
df = embeddings.join(clusters).join(metadata).join(firmas[['intensidad_log', 'ratio_violencia']])

print(f"Hexágonos: {len(df)}")
print(f"Columnas: {df.shape[1]}")

Hexágonos: 1061
Columnas: 22


#### Implementación

In [8]:
# ============================================================================
# PASO 1: UMAP
# ============================================================================

emb_cols = [c for c in embeddings.columns if c.startswith('emb_')]
X_emb = df[emb_cols].values

# UMAP con parámetros robustos
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)
umap_2d = reducer.fit_transform(X_emb)
df['umap_x'] = umap_2d[:, 0]
df['umap_y'] = umap_2d[:, 1]
print(f"UMAP completado: {umap_2d.shape}")

# ============================================================================
# PASO 2: t-SNE
# ============================================================================

tsne = TSNE(
    n_components=2,
    perplexity=30,
    max_iter=1000,
    random_state=42,
    init='pca'
)
tsne_2d = tsne.fit_transform(X_emb)
df['tsne_x'] = tsne_2d[:, 0]
df['tsne_y'] = tsne_2d[:, 1]
print(f"t-SNE completado: {tsne_2d.shape}")

UMAP completado: (1061, 2)
  t-SNE completado: (1061, 2)


In [9]:
# ============================================================================
# PASO 3: Visualizaciones abstractas con Plotly (interactivas)
# ============================================================================

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preparar datos para hover
df['cluster_str'] = df['cluster_kmeans'].astype(str)
df['intensidad_real'] = np.expm1(df['intensidad_log']).astype(int)

# Delito dominante por hexágono
delito_cols = [c for c in firmas.columns if c.startswith('delito_')]
df['delito_dominante'] = firmas[delito_cols].idxmax(axis=1).str.replace('delito_', '').str.replace('_', ' ').str.upper()

hover_template = (
    "<b>%{customdata[0]}</b><br>"
    "Colonia: %{customdata[1]}<br>"
    "Cluster: %{customdata[2]}<br>"
    "Registros: %{customdata[3]:,}<br>"
    "Ratio violencia: %{customdata[4]:.3f}<br>"
    "Delito dominante: %{customdata[5]}<br>"
    "<extra></extra>"
)
customdata = df[['alcaldia_dominante', 'colonia_dominante', 'cluster_str',
                  'intensidad_real', 'ratio_violencia', 'delito_dominante']].values

# --- 3a. UMAP coloreado por cluster ---
fig_umap_cluster = px.scatter(
    df, x='umap_x', y='umap_y',
    color='cluster_str',
    hover_data=['alcaldia_dominante', 'colonia_dominante', 'intensidad_real', 'ratio_violencia'],
    title='Espacio Latente UMAP — Clusters K-Means (GAE óptimo)',
    labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2', 'cluster_str': 'Cluster'},
    width=900, height=700,
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig_umap_cluster.update_traces(marker=dict(size=6, opacity=0.8))
fig_umap_cluster.update_layout(legend_title_text='Cluster')
fig_umap_cluster.write_html('../visualizations/vis_umap_clusters.html')
print(f"/visualizations/vis_umap_clusters.html")

# --- 3b. UMAP coloreado por alcaldía ---
fig_umap_alc = px.scatter(
    df, x='umap_x', y='umap_y',
    color='alcaldia_dominante',
    hover_data=['colonia_dominante', 'cluster_str', 'intensidad_real', 'ratio_violencia'],
    title='Espacio Latente UMAP — Por Alcaldía',
    labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2', 'alcaldia_dominante': 'Alcaldía'},
    width=900, height=700,
    color_discrete_sequence=px.colors.qualitative.Dark24
)
fig_umap_alc.update_traces(marker=dict(size=6, opacity=0.8))
fig_umap_alc.write_html('../visualizations/vis_umap_alcaldias.html')
print(f"/visualizations/vis_umap_alcaldias.html")

# --- 3c. UMAP coloreado por ratio de violencia ---
fig_umap_viol = px.scatter(
    df, x='umap_x', y='umap_y',
    color='ratio_violencia',
    hover_data=['alcaldia_dominante', 'colonia_dominante', 'cluster_str', 'intensidad_real'],
    title='Espacio Latente UMAP — Ratio de Violencia',
    labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2', 'ratio_violencia': 'Ratio Violencia'},
    width=900, height=700,
    color_continuous_scale='RdYlBu_r'  # Rojo = más violencia
)
fig_umap_viol.update_traces(marker=dict(size=6, opacity=0.8))
fig_umap_viol.write_html('../visualizations/vis_umap_violencia.html')
print(f"/visualizations/vis_umap_violencia.html")

# --- 3d. UMAP coloreado por intensidad ---
fig_umap_int = px.scatter(
    df, x='umap_x', y='umap_y',
    color='intensidad_log',
    hover_data=['alcaldia_dominante', 'colonia_dominante', 'cluster_str', 'intensidad_real'],
    title='Espacio Latente UMAP — Intensidad (log registros)',
    labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2', 'intensidad_log': 'Intensidad (log)'},
    width=900, height=700,
    color_continuous_scale='Viridis'
)
fig_umap_int.update_traces(marker=dict(size=6, opacity=0.8))
fig_umap_int.write_html('../visualizations/vis_umap_intensidad.html')
print(f"/visualizations/vis_umap_intensidad.html")

# --- 3e. t-SNE coloreado por cluster ---
fig_tsne = px.scatter(
    df, x='tsne_x', y='tsne_y',
    color='cluster_str',
    hover_data=['alcaldia_dominante', 'colonia_dominante', 'intensidad_real', 'ratio_violencia'],
    title='Espacio Latente t-SNE — Clusters K-Means (GAE óptimo)',
    labels={'tsne_x': 't-SNE 1', 'tsne_y': 't-SNE 2', 'cluster_str': 'Cluster'},
    width=900, height=700,
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig_tsne.update_traces(marker=dict(size=6, opacity=0.8))
fig_tsne.write_html('../visualizations/vis_tsne_clusters.html')
print(f"/visualizations/vis_tsne_clusters.html")

/visualizations/vis_umap_clusters.html
/visualizations/vis_umap_alcaldias.html
/visualizations/vis_umap_violencia.html
/visualizations/vis_umap_intensidad.html
/visualizations/vis_tsne_clusters.html


In [12]:
# ============================================================================
# PASO 4: Visualizaciones geográficas con Folium
# ============================================================================

# Centro de CDMX
CDMX_CENTER = [19.38, -99.14]

# Función para obtener bordes del hexágono H3
def h3_to_polygon(h3_id):
    """Retorna lista de [lat, lng] para el borde del hexágono."""
    boundary = h3.cell_to_boundary(h3_id)
    # h3 retorna (lat, lng), folium necesita [lat, lng]
    return [[lat, lng] for lat, lng in boundary]

# --- 4a. Mapa de clusters ---

# Paleta de colores para 15 clusters
cluster_colors = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe',
    '#008080', '#e6beff', '#9a6324', '#800000', '#aaffc3'
]

m_clusters = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB positron')

for h3_id, row in df.iterrows():
    polygon = h3_to_polygon(h3_id)
    cluster = int(row['cluster_kmeans'])
    color = cluster_colors[cluster % len(cluster_colors)]

    popup_html = f"""
    <b>Cluster {cluster}</b><br>
    Alcaldía: {row['alcaldia_dominante']}<br>
    Colonia: {row['colonia_dominante']}<br>
    Registros: {int(row['intensidad_real']):,}<br>
    Ratio violencia: {row['ratio_violencia']:.3f}<br>
    Delito dominante: {row['delito_dominante']}
    """

    folium.Polygon(
        locations=polygon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"Cluster {cluster} | {row['alcaldia_dominante']}"
    ).add_to(m_clusters)

# Leyenda
legend_html = '<div style="position:fixed;bottom:50px;left:50px;z-index:1000;background:white;padding:10px;border-radius:5px;border:1px solid grey;">'
legend_html += '<b>Clusters K-Means</b><br>'
for i in range(min(15, len(df['cluster_kmeans'].unique()))):
    legend_html += f'<i style="background:{cluster_colors[i]};width:12px;height:12px;display:inline-block;margin-right:5px;"></i> Cluster {i}<br>'
legend_html += '</div>'
m_clusters.get_root().html.add_child(folium.Element(legend_html))

m_clusters.save('../visualizations/mapa_clusters.html')
print(f"/visualizations/mapa_clusters.html")

# --- 4b. Mapa de ratio de violencia ---

m_violencia = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB dark_matter')

# Colormap: azul (baja violencia) → rojo (alta violencia)
vmin, vmax = df['ratio_violencia'].quantile(0.05), df['ratio_violencia'].quantile(0.95)
colormap_viol = cm.LinearColormap(
    colors=['#2166ac', '#67a9cf', '#fddbc7', '#ef8a62', '#b2182b'],
    vmin=vmin, vmax=vmax,
    caption='Ratio de Violencia'
)

for h3_id, row in df.iterrows():
    polygon = h3_to_polygon(h3_id)
    ratio = row['ratio_violencia']
    color = colormap_viol(min(max(ratio, vmin), vmax))

    popup_html = f"""
    <b>Ratio violencia: {ratio:.3f}</b><br>
    Alcaldía: {row['alcaldia_dominante']}<br>
    Colonia: {row['colonia_dominante']}<br>
    Registros: {int(row['intensidad_real']):,}<br>
    Cluster: {int(row['cluster_kmeans'])}
    """

    folium.Polygon(
        locations=polygon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"Violencia: {ratio:.3f} | {row['alcaldia_dominante']}"
    ).add_to(m_violencia)

colormap_viol.add_to(m_violencia)
m_violencia.save('../visualizations/mapa_violencia.html')
print(f"/visualizations/mapa_violencia.html")

# --- 4c. Mapa de intensidad ---

m_intensidad = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB positron')

vmin_int = df['intensidad_log'].quantile(0.05)
vmax_int = df['intensidad_log'].quantile(0.95)
colormap_int = cm.LinearColormap(
    colors=['#ffffcc', '#a1dab4', '#41b6c4', '#2c7fb8', '#253494'],
    vmin=vmin_int, vmax=vmax_int,
    caption='Intensidad (log registros)'
)

for h3_id, row in df.iterrows():
    polygon = h3_to_polygon(h3_id)
    intens = row['intensidad_log']
    color = colormap_int(min(max(intens, vmin_int), vmax_int))

    popup_html = f"""
    <b>Registros: {int(row['intensidad_real']):,}</b><br>
    Alcaldía: {row['alcaldia_dominante']}<br>
    Colonia: {row['colonia_dominante']}<br>
    Ratio violencia: {row['ratio_violencia']:.3f}<br>
    Cluster: {int(row['cluster_kmeans'])}
    """

    folium.Polygon(
        locations=polygon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{int(row['intensidad_real']):,} registros | {row['alcaldia_dominante']}"
    ).add_to(m_intensidad)

colormap_int.add_to(m_intensidad)
m_intensidad.save('../visualizations/mapa_intensidad.html')
print(f"/visualizations/mapa_intensidad.html")

# --- 4d. Mapa de delito dominante ---

m_delito = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB positron')

# Colores por delito dominante
delitos_unicos = df['delito_dominante'].unique()
delito_color_map = {}
palette = px.colors.qualitative.Set3 + px.colors.qualitative.Pastel1
for i, d in enumerate(delitos_unicos):
    delito_color_map[d] = palette[i % len(palette)]

for h3_id, row in df.iterrows():
    polygon = h3_to_polygon(h3_id)
    delito = row['delito_dominante']
    color = delito_color_map[delito]

    popup_html = f"""
    <b>{delito}</b><br>
    Alcaldía: {row['alcaldia_dominante']}<br>
    Colonia: {row['colonia_dominante']}<br>
    Registros: {int(row['intensidad_real']):,}<br>
    Ratio violencia: {row['ratio_violencia']:.3f}
    """

    folium.Polygon(
        locations=polygon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{delito} | {row['alcaldia_dominante']}"
    ).add_to(m_delito)

m_delito.save('../visualizations/mapa_delito_dominante.html')
print(f"/visualizations/mapa_delito_dominante.html")

/visualizations/mapa_clusters.html
/visualizations/mapa_violencia.html
/visualizations/mapa_intensidad.html
/visualizations/mapa_delito_dominante.html


In [14]:
# ============================================================================
# PASO 5: Exportar coordenadas de proyección
# ============================================================================

proyecciones = df[['umap_x', 'umap_y', 'tsne_x', 'tsne_y']].copy()
proyecciones.index.name = 'h3_id'
proyecciones.to_csv('../data/vis_results/proyecciones_2d.csv', encoding='utf-8-sig')
print(f"/data/vis_results/proyecciones_2d.csv")

/data/vis_results/proyecciones_2d.csv


# TERMINA NICOLE

# EMPIEZA LUISMI

#### Dashboard completo
Combina las 9 visualizaciones en un solo HTML con navegación por tabs.

In [16]:
embeddings = pd.read_csv("../data/results/embeddings_h3_optimal.csv", index_col='h3_id')
clusters = pd.read_csv("../data/results/clusters_h3_optimal.csv", index_col='h3_id')
metadata = pd.read_csv("../data/auxiliar/h3_metadata.csv", index_col='h3_id')
firmas = pd.read_csv("../data/auxiliar/firmas_h3.csv", index_col='h3_id')
proyecciones = pd.read_csv("../data/vis_results/proyecciones_2d.csv", index_col='h3_id')

df = embeddings.join(clusters).join(metadata).join(firmas[['intensidad_log', 'ratio_violencia']]).join(proyecciones)
df['intensidad_real'] = np.expm1(df['intensidad_log']).astype(int)
df['cluster'] = df['cluster_kmeans'].astype(int)

delito_cols = [c for c in firmas.columns if c.startswith('delito_')]
df['delito_dominante'] = firmas[delito_cols].idxmax(axis=1).str.replace('delito_', '').str.replace('_', ' ').str.upper()

# Top 3 delitos por hexágono
def top3_delitos(row):
    vals = row[delito_cols].sort_values(ascending=False).head(3)
    return ' | '.join([f"{c.replace('delito_','').replace('_',' ').upper()}: {v*100:.0f}%" for c, v in vals.items()])

df['top3_delitos'] = firmas.apply(top3_delitos, axis=1)

# Generar polígonos H3
hex_polygons = {}
for h3_id in df.index:
    boundary = h3.cell_to_boundary(h3_id)
    hex_polygons[h3_id] = [[lat, lng] for lat, lng in boundary]

# ============================================================================
# Guardar en JSON para consumir en el front
# ============================================================================

hexagons_data = []
for h3_id, row in df.iterrows():
    hexagons_data.append({
        'id': h3_id,
        'polygon': hex_polygons[h3_id],
        'lat': row['lat_centro'],
        'lng': row['lon_centro'],
        'cluster': int(row['cluster']),
        'alcaldia': row['alcaldia_dominante'],
        'colonia': str(row['colonia_dominante']),
        'registros': int(row['intensidad_real']),
        'intensidad': round(row['intensidad_log'], 2),
        'violencia': round(row['ratio_violencia'], 3),
        'delito_dom': row['delito_dominante'],
        'top3': row['top3_delitos'],
        'umap_x': round(row['umap_x'], 4),
        'umap_y': round(row['umap_y'], 4),
        'tsne_x': round(row['tsne_x'], 4),
        'tsne_y': round(row['tsne_y'], 4),
        'hdbscan': int(row['cluster_hdbscan']),
    })

data_json = json.dumps(hexagons_data)

# ============================================================================
# Generar HTML
# ============================================================================

html = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Urban Crime CDMX — Dashboard</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
  * {{ margin: 0; padding: 0; box-sizing: border-box; }}
  body {{ font-family: 'Segoe UI', system-ui, -apple-system, sans-serif; background: #0a0a0f; color: #e0e0e0; }}

  /* Header */
  .header {{
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    padding: 16px 24px;
    border-bottom: 1px solid #2a2a4a;
    display: flex;
    align-items: center;
    justify-content: space-between;
  }}
  .header h1 {{ font-size: 18px; font-weight: 600; color: #fff; }}
  .header .subtitle {{ font-size: 12px; color: #888; margin-top: 2px; }}

  /* Tabs */
  .tabs {{
    display: flex;
    background: #12121f;
    border-bottom: 1px solid #2a2a4a;
    overflow-x: auto;
  }}
  .tab {{
    padding: 10px 18px;
    cursor: pointer;
    font-size: 13px;
    color: #888;
    border-bottom: 2px solid transparent;
    white-space: nowrap;
    transition: all 0.2s;
  }}
  .tab:hover {{ color: #ccc; background: #1a1a2e; }}
  .tab.active {{ color: #64b5f6; border-bottom-color: #64b5f6; }}

  /* Main content */
  .content {{ display: flex; height: calc(100vh - 95px); }}

  /* Map */
  #map {{ flex: 1; background: #0d1117; }}

  /* Scatter panel */
  #scatter-panel {{
    width: 380px;
    background: #12121f;
    border-left: 1px solid #2a2a4a;
    display: flex;
    flex-direction: column;
  }}
  #scatter-header {{
    padding: 12px 16px;
    border-bottom: 1px solid #2a2a4a;
    display: flex;
    justify-content: space-between;
    align-items: center;
  }}
  #scatter-header h3 {{ font-size: 13px; font-weight: 600; }}
  #scatter-header select {{
    background: #1a1a2e;
    color: #ccc;
    border: 1px solid #2a2a4a;
    padding: 4px 8px;
    border-radius: 4px;
    font-size: 12px;
  }}
  #scatter-canvas {{ flex: 1; cursor: crosshair; }}

  /* Info panel */
  #info-panel {{
    position: absolute;
    bottom: 20px;
    left: 20px;
    z-index: 1000;
    background: rgba(18, 18, 31, 0.95);
    border: 1px solid #2a2a4a;
    border-radius: 8px;
    padding: 14px 18px;
    min-width: 280px;
    max-width: 340px;
    font-size: 12px;
    line-height: 1.6;
    display: none;
  }}
  #info-panel h4 {{ color: #64b5f6; margin-bottom: 6px; font-size: 14px; }}
  #info-panel .label {{ color: #888; }}
  #info-panel .value {{ color: #e0e0e0; font-weight: 500; }}

  /* Stats bar */
  .stats-bar {{
    position: absolute;
    top: 100px;
    right: 400px;
    z-index: 1000;
    background: rgba(18, 18, 31, 0.9);
    border: 1px solid #2a2a4a;
    border-radius: 8px;
    padding: 10px 14px;
    font-size: 11px;
  }}
  .stats-bar .stat {{ display: inline-block; margin-right: 16px; }}
  .stats-bar .stat-value {{ color: #64b5f6; font-weight: 600; font-size: 14px; }}

  /* Legend */
  #legend {{
    position: absolute;
    bottom: 20px;
    right: 400px;
    z-index: 1000;
    background: rgba(18, 18, 31, 0.95);
    border: 1px solid #2a2a4a;
    border-radius: 8px;
    padding: 10px 14px;
    font-size: 11px;
    max-height: 300px;
    overflow-y: auto;
  }}
  .legend-item {{ display: flex; align-items: center; margin: 3px 0; }}
  .legend-color {{ width: 14px; height: 14px; border-radius: 3px; margin-right: 8px; flex-shrink: 0; }}

  /* Gradient legend */
  .gradient-legend {{
    display: flex;
    align-items: center;
    margin-top: 6px;
  }}
  .gradient-bar {{
    width: 150px;
    height: 12px;
    border-radius: 3px;
    margin: 0 8px;
  }}
</style>
</head>
<body>

<div class="header">
  <div>
    <h1>Urban Crime CDMX</h1>
    <div class="subtitle">1,061 hexágonos H3 · GAE embeddings · 2016–2024</div>
  </div>
</div>

<div class="tabs">
  <div class="tab active" data-mode="clusters">Clusters</div>
  <div class="tab" data-mode="violencia">Violencia</div>
  <div class="tab" data-mode="intensidad">Intensidad</div>
  <div class="tab" data-mode="delito">Delito Dominante</div>
  <div class="tab" data-mode="alcaldia">Alcaldía</div>
  <div class="tab" data-mode="hdbscan">HDBSCAN</div>
</div>

<div class="content">
  <div id="map"></div>
  <div id="scatter-panel">
    <div id="scatter-header">
      <h3>Espacio Latente</h3>
      <select id="scatter-method">
        <option value="umap">UMAP</option>
        <option value="tsne">t-SNE</option>
      </select>
    </div>
    <canvas id="scatter-canvas"></canvas>
  </div>
</div>

<div id="info-panel"></div>
<div id="legend"></div>

<div class="stats-bar">
  <span class="stat"><span class="label">Hexágonos: </span><span class="stat-value">1,061</span></span>
  <span class="stat"><span class="label">Clusters: </span><span class="stat-value">15</span></span>
  <span class="stat"><span class="label">Modelo: </span><span class="stat-value">GAE</span></span>
</div>

<script>
const DATA = {data_json};

// Colors
const CLUSTER_COLORS = [
  '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
  '#911eb4','#46f0f0','#f032e6','#bcf60c','#fabebe',
  '#008080','#e6beff','#9a6324','#800000','#aaffc3'
];

const ALCALDIA_COLORS = {{}};
const alcaldias = [...new Set(DATA.map(d => d.alcaldia))].sort();
const ALC_PALETTE = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4','#46f0f0','#f032e6',
  '#bcf60c','#fabebe','#008080','#e6beff','#9a6324','#800000','#aaffc3','#ffe119','#000075'];
alcaldias.forEach((a, i) => ALCALDIA_COLORS[a] = ALC_PALETTE[i % ALC_PALETTE.length]);

const DELITO_COLORS = {{}};
const delitos = [...new Set(DATA.map(d => d.delito_dom))].sort();
delitos.forEach((d, i) => DELITO_COLORS[d] = ALC_PALETTE[i % ALC_PALETTE.length]);

// State
let currentMode = 'clusters';
let selectedHex = null;
let hexLayers = {{}};

// Map
const map = L.map('map', {{
  center: [19.38, -99.14],
  zoom: 11,
  zoomControl: true,
}});

L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
  attribution: '© CartoDB',
  maxZoom: 18
}}).addTo(map);

// Interpolate color for continuous scales
function interpolateColor(value, min, max, colors) {{
  const t = Math.max(0, Math.min(1, (value - min) / (max - min)));
  const idx = t * (colors.length - 1);
  const lo = Math.floor(idx);
  const hi = Math.min(lo + 1, colors.length - 1);
  const f = idx - lo;
  const c1 = hexToRgb(colors[lo]);
  const c2 = hexToRgb(colors[hi]);
  const r = Math.round(c1.r + f * (c2.r - c1.r));
  const g = Math.round(c1.g + f * (c2.g - c1.g));
  const b = Math.round(c1.b + f * (c2.b - c1.b));
  return `rgb(${{r}},${{g}},${{b}})`;
}}

function hexToRgb(hex) {{
  const r = parseInt(hex.slice(1,3), 16);
  const g = parseInt(hex.slice(3,5), 16);
  const b = parseInt(hex.slice(5,7), 16);
  return {{r, g, b}};
}}

// Violencia scale
const violMin = 0.03, violMax = 0.35;
const violColors = ['#2166ac','#67a9cf','#fddbc7','#ef8a62','#b2182b'];

// Intensidad scale
const intMin = 3.5, intMax = 9.5;
const intColors = ['#ffffcc','#a1dab4','#41b6c4','#2c7fb8','#253494'];

function getHexColor(d, mode) {{
  switch(mode) {{
    case 'clusters': return CLUSTER_COLORS[d.cluster % CLUSTER_COLORS.length];
    case 'violencia': return interpolateColor(d.violencia, violMin, violMax, violColors);
    case 'intensidad': return interpolateColor(d.intensidad, intMin, intMax, intColors);
    case 'delito': return DELITO_COLORS[d.delito_dom] || '#666';
    case 'alcaldia': return ALCALDIA_COLORS[d.alcaldia] || '#666';
    case 'hdbscan': return d.hdbscan === -1 ? '#333' : CLUSTER_COLORS[d.hdbscan % CLUSTER_COLORS.length];
    default: return '#666';
  }}
}}

// Draw hexagons
function drawHexagons() {{
  Object.values(hexLayers).forEach(l => map.removeLayer(l));
  hexLayers = {{}};

  DATA.forEach(d => {{
    const color = getHexColor(d, currentMode);
    const poly = L.polygon(d.polygon, {{
      color: color,
      weight: 0.8,
      fillColor: color,
      fillOpacity: 0.65,
    }});

    poly.on('mouseover', function() {{
      this.setStyle({{ weight: 2.5, fillOpacity: 0.9 }});
      showInfo(d);
    }});
    poly.on('mouseout', function() {{
      if (selectedHex !== d.id) {{
        this.setStyle({{ weight: 0.8, fillOpacity: 0.65 }});
      }}
    }});
    poly.on('click', function() {{
      selectedHex = d.id;
      showInfo(d);
      highlightScatter(d);
    }});

    poly.addTo(map);
    hexLayers[d.id] = poly;
  }});

  updateLegend();
}}

function showInfo(d) {{
  const panel = document.getElementById('info-panel');
  panel.style.display = 'block';
  panel.innerHTML = `
    <h4>${{d.alcaldia}}</h4>
    <span class="label">Colonia:</span> <span class="value">${{d.colonia}}</span><br>
    <span class="label">Cluster:</span> <span class="value">${{d.cluster}}</span>
    <span class="label" style="margin-left:12px">HDBSCAN:</span> <span class="value">${{d.hdbscan === -1 ? 'Ruido' : d.hdbscan}}</span><br>
    <span class="label">Registros:</span> <span class="value">${{d.registros.toLocaleString()}}</span><br>
    <span class="label">Ratio violencia:</span> <span class="value">${{d.violencia.toFixed(3)}}</span><br>
    <span class="label">Delito dominante:</span> <span class="value">${{d.delito_dom}}</span><br>
    <hr style="border-color:#2a2a4a;margin:6px 0">
    <span class="label">Top 3:</span><br>
    <span class="value" style="font-size:11px">${{d.top3}}</span>
  `;
}}

function updateLegend() {{
  const legend = document.getElementById('legend');
  let html = '';

  if (currentMode === 'clusters') {{
    html = '<b>Clusters K-Means</b><br>';
    for (let i = 0; i < 15; i++) {{
      const count = DATA.filter(d => d.cluster === i).length;
      html += `<div class="legend-item"><div class="legend-color" style="background:${{CLUSTER_COLORS[i]}}"></div>Cluster ${{i}} (${{count}})</div>`;
    }}
  }} else if (currentMode === 'hdbscan') {{
    html = '<b>Clusters HDBSCAN</b><br>';
    const hdbClusters = [...new Set(DATA.map(d => d.hdbscan))].sort((a,b) => a-b);
    hdbClusters.forEach(c => {{
      const count = DATA.filter(d => d.hdbscan === c).length;
      const color = c === -1 ? '#333' : CLUSTER_COLORS[c % CLUSTER_COLORS.length];
      const label = c === -1 ? 'Ruido' : `Cluster ${{c}}`;
      html += `<div class="legend-item"><div class="legend-color" style="background:${{color}}"></div>${{label}} (${{count}})</div>`;
    }});
  }} else if (currentMode === 'violencia') {{
    html = '<b>Ratio de Violencia</b><br>';
    html += '<div class="gradient-legend"><span>0.03</span>';
    html += '<div class="gradient-bar" style="background:linear-gradient(to right,#2166ac,#67a9cf,#fddbc7,#ef8a62,#b2182b)"></div>';
    html += '<span>0.35</span></div>';
  }} else if (currentMode === 'intensidad') {{
    html = '<b>Intensidad (log)</b><br>';
    html += '<div class="gradient-legend"><span>30</span>';
    html += '<div class="gradient-bar" style="background:linear-gradient(to right,#ffffcc,#a1dab4,#41b6c4,#2c7fb8,#253494)"></div>';
    html += '<span>13k+</span></div>';
  }} else if (currentMode === 'alcaldia') {{
    html = '<b>Alcaldía</b><br>';
    alcaldias.forEach(a => {{
      const count = DATA.filter(d => d.alcaldia === a).length;
      html += `<div class="legend-item"><div class="legend-color" style="background:${{ALCALDIA_COLORS[a]}}"></div>${{a}} (${{count}})</div>`;
    }});
  }} else if (currentMode === 'delito') {{
    html = '<b>Delito Dominante</b><br>';
    const sorted = [...new Set(DATA.map(d => d.delito_dom))].sort();
    sorted.forEach(d => {{
      const count = DATA.filter(h => h.delito_dom === d).length;
      if (count > 5) {{
        html += `<div class="legend-item"><div class="legend-color" style="background:${{DELITO_COLORS[d]}}"></div>${{d}} (${{count}})</div>`;
      }}
    }});
  }}

  legend.innerHTML = html;
}}

// Scatter plot
const canvas = document.getElementById('scatter-canvas');
const ctx = canvas.getContext('2d');
let scatterMethod = 'umap';

function resizeCanvas() {{
  canvas.width = canvas.parentElement.clientWidth;
  canvas.height = canvas.parentElement.clientHeight - 45;
}}

function drawScatter() {{
  resizeCanvas();
  const w = canvas.width, h = canvas.height;
  const pad = 30;

  ctx.fillStyle = '#12121f';
  ctx.fillRect(0, 0, w, h);

  const xKey = scatterMethod === 'umap' ? 'umap_x' : 'tsne_x';
  const yKey = scatterMethod === 'umap' ? 'umap_y' : 'tsne_y';

  const xs = DATA.map(d => d[xKey]);
  const ys = DATA.map(d => d[yKey]);
  const xMin = Math.min(...xs), xMax = Math.max(...xs);
  const yMin = Math.min(...ys), yMax = Math.max(...ys);

  const scaleX = v => pad + (v - xMin) / (xMax - xMin) * (w - 2*pad);
  const scaleY = v => h - pad - (v - yMin) / (yMax - yMin) * (h - 2*pad);

  DATA.forEach(d => {{
    const x = scaleX(d[xKey]);
    const y = scaleY(d[yKey]);
    const color = getHexColor(d, currentMode);

    ctx.beginPath();
    ctx.arc(x, y, 3.5, 0, Math.PI * 2);
    ctx.fillStyle = color;
    ctx.globalAlpha = 0.8;
    ctx.fill();
    ctx.globalAlpha = 1;

    // Store position for hit detection
    d._sx = x;
    d._sy = y;
  }});

  // Axes labels
  ctx.fillStyle = '#555';
  ctx.font = '11px system-ui';
  ctx.textAlign = 'center';
  const label = scatterMethod === 'umap' ? 'UMAP' : 't-SNE';
  ctx.fillText(label + ' 1', w/2, h - 5);
  ctx.save();
  ctx.translate(12, h/2);
  ctx.rotate(-Math.PI/2);
  ctx.fillText(label + ' 2', 0, 0);
  ctx.restore();
}}

function highlightScatter(d) {{
  drawScatter();
  if (!d) return;
  const xKey = scatterMethod === 'umap' ? 'umap_x' : 'tsne_x';
  const yKey = scatterMethod === 'umap' ? 'umap_y' : 'tsne_y';

  ctx.beginPath();
  ctx.arc(d._sx, d._sy, 8, 0, Math.PI * 2);
  ctx.strokeStyle = '#fff';
  ctx.lineWidth = 2;
  ctx.stroke();
}}

// Canvas hover/click
canvas.addEventListener('mousemove', function(e) {{
  const rect = canvas.getBoundingClientRect();
  const mx = e.clientX - rect.left;
  const my = e.clientY - rect.top;

  let closest = null, minDist = 20;
  DATA.forEach(d => {{
    if (d._sx === undefined) return;
    const dist = Math.sqrt((d._sx - mx)**2 + (d._sy - my)**2);
    if (dist < minDist) {{ minDist = dist; closest = d; }}
  }});

  if (closest) {{
    canvas.style.cursor = 'pointer';
    showInfo(closest);
  }} else {{
    canvas.style.cursor = 'crosshair';
  }}
}});

canvas.addEventListener('click', function(e) {{
  const rect = canvas.getBoundingClientRect();
  const mx = e.clientX - rect.left;
  const my = e.clientY - rect.top;

  let closest = null, minDist = 20;
  DATA.forEach(d => {{
    if (d._sx === undefined) return;
    const dist = Math.sqrt((d._sx - mx)**2 + (d._sy - my)**2);
    if (dist < minDist) {{ minDist = dist; closest = d; }}
  }});

  if (closest) {{
    selectedHex = closest.id;
    showInfo(closest);
    map.setView([closest.lat, closest.lng], 14);
    highlightScatter(closest);
    // Highlight on map
    Object.values(hexLayers).forEach(l => l.setStyle({{ weight: 0.8, fillOpacity: 0.65 }}));
    if (hexLayers[closest.id]) {{
      hexLayers[closest.id].setStyle({{ weight: 3, fillOpacity: 0.95 }});
    }}
  }}
}});

// Tab switching
document.querySelectorAll('.tab').forEach(tab => {{
  tab.addEventListener('click', function() {{
    document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
    this.classList.add('active');
    currentMode = this.dataset.mode;
    drawHexagons();
    drawScatter();
  }});
}});

// Scatter method switching
document.getElementById('scatter-method').addEventListener('change', function() {{
  scatterMethod = this.value;
  drawScatter();
}});

// Window resize
window.addEventListener('resize', () => {{ drawScatter(); }});

// Init
drawHexagons();
setTimeout(drawScatter, 100);

</script>
</body>
</html>"""

# Guardar
with open('../visualizations/dashboard_urban_crime.html', 'w', encoding='utf-8') as f:
    f.write(html)

# TERMINA LUISMI